# Травми й бої

Бої формульного розв'язувача (D42) і травми (D37) за ботами (tech 6.2, «вплив травм»). Метрика спіралі смерті (tech 4.6): тиждень, коли шкала травми вперше дійшла до «важкої» частки максимуму, і скільки після цього культивація стояла.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from simruns import BOTS, label, load, ranks

RUNS, SEED_FROM = 1000, 1  # per bot; data regenerates by itself when Core, Sim.Cli or content changes
data = load(runs=RUNS, seed_from=SEED_FROM)
runs = data.runs
pd.options.display.float_format = "{:.2f}".format
plt.rcParams["figure.figsize"] = (10, 4)


def named(df, axis=0):
    """Replaces content IDs in the index (axis=0) or columns (axis=1) with Ukrainian names."""
    return df.rename(index=label) if axis == 0 else df.rename(columns=label)


def per_run(long, value, by):
    """Mean of value per run for each bot, counting runs where it never happened as zero."""
    total = long.pivot_table(index=by, columns="bot", values=value, aggfunc="sum", fill_value=0)
    return total.div(runs.groupby("bot").size(), axis=1).reindex(columns=list(BOTS), fill_value=0)


## Бої за ботами

Боїв на забіг, частка забігів хоч з одним боєм, частки результатів серед усіх боїв і осяяння з боїв.

In [ ]:
g = runs.groupby("bot")
fights = g.fights.sum()
pd.DataFrame({
    "боїв на забіг": g.fights.mean(),
    "забігів із боєм": g.fights.apply(lambda f: (f > 0).mean()),
    "перемога": g.won.sum() / fights,
    "втеча": g.fled.sum() / fights,
    "загибель": g.died.sum() / fights,
    "осяяння з боїв": g.combat_insight.mean(),
}).reindex(list(BOTS))

## Бої за групами ворогів

Усі бої з групою за бот і частки результатів. Решта до 1 — бої без розв'язки (D42: відступ).

In [ ]:
cg = data.combat_groups.groupby(["group", "bot"])[["won", "fled", "died"]].sum()
total = cg.sum(axis=1)
named(cg.div(total, axis=0).assign(боїв=total))

## Пікові травми

Найвище значення шкали кожної травми за забіг (забіги без травми — нуль).

In [ ]:
peak = data.injuries.pivot_table(index=["seed", "bot"], columns="injury", values="peak", fill_value=0)
peak = peak.reindex(pd.MultiIndex.from_frame(runs[["seed", "bot"]]), fill_value=0)
named(peak.groupby("bot").describe(percentiles=[0.5, 0.9]).T.unstack(0).swaplevel(axis=1).sort_index(axis=1))

In [ ]:
fig, axes = plt.subplots(1, len(peak.columns), figsize=(5 * len(peak.columns), 4))
for ax, injury in zip(axes, peak.columns):
    for bot in BOTS:
        values = peak.xs(bot, level="bot")[injury]
        ax.hist(values[values > 0], bins=30, histtype="step", linewidth=1.5, label=bot)
    ax.set(title=label(injury), xlabel="пік шкали (без нулів)", ylabel="забігів")
axes[0].legend();

## Спіраль смерті

Частка забігів, де травма ставала важкою; серед них — тижні без прогресу культивації після цього і частка тих, хто ще просунувся.

In [ ]:
heavy = runs[runs.first_heavy_week.notna()]
hg = heavy.groupby("bot")
pd.DataFrame({
    "забігів із важкою травмою": (heavy.groupby("bot").size() / runs.groupby("bot").size()).fillna(0),
    "тижнів стояння, медіана": hg.weeks_stalled_after_heavy.median(),
    "p90": hg.weeks_stalled_after_heavy.quantile(0.9),
    "просунулись після": hg.progressed_after_heavy.mean(),
}).reindex(list(BOTS))

## Лікування

Тижні на лікування за забіг.

In [ ]:
runs.groupby("bot").healing_weeks.describe(percentiles=[0.5, 0.9]).reindex(list(BOTS))